# Chapter 9 examples: nonlinear dynamics

This notebook runs the examples of
[Chapter 9](https://thduynguyen.github.io/gtsam/chapter09): iLQR and DDP on
a robot with a weak motor, the same problem as a MAP trajectory optimization
with GTSAM's Levenberg-Marquardt optimizer, and model predictive control
under noise.

GTSAM Copyright 2010, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/thduynguyen/gtsam/blob/feature/semiringfactor/gtsam/semiring/doc/chapter09_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    # The semiring module is not in a released GTSAM yet, so install a wheel
    # built from the feature branch.
    %pip install --quiet numpy pyparsing plotly
    %pip install --quiet --no-index --no-deps --find-links https://github.com/thduynguyen/gtsam/releases/expanded_assets/semiring-wheels gtsam-develop
except ImportError:
    pass  # Not running on Colab, do nothing

In [2]:
import numpy as np
from scipy.optimize import least_squares, minimize

import gtsam
from gtsam import CustomFactor, HessianFactor, JacobianFactor, Ordering
from gtsam import SemiringGaussianFactor, VectorValues, noiseModel
from gtsam.symbol_shorthand import U, X

np.set_printoptions(precision=4, suppress=True)

## The weak motor (Section 1)

The robot of the line example, with a motor that saturates: a command $u$
moves it by $\tanh(u)$, never more than 1. It starts at $x_0 = 3$ and makes
four moves. The rewards are those of the line.

In [3]:
T = 4  # moves
x_start = 3.0


def f(x, u):
    """The mean of the dynamics: x' = f(x, u) + noise."""
    return x + np.tanh(u)


def f_u(u):
    """The derivative of f with respect to u; the one with respect to x is 1."""
    return 1 - np.tanh(u) ** 2


def rollout(controls, x0):
    """The states visited when the controls are applied without noise."""
    states = [x0]
    for u in controls:
        states.append(f(states[-1], u))
    return np.array(states)


def total_reward(controls, x0):
    """The return of the noise-free rollout."""
    states = rollout(controls, x0)
    return -(np.sum(states[:-1] ** 2) + np.sum(np.square(controls))
             + states[-1] ** 2)


print("doing nothing:", total_reward(np.zeros(T), x_start))
assert np.isclose(total_reward(np.zeros(T), x_start), -45.0)

doing nothing: -45.0


## Stage 1 with the module: linearize, then eliminate (Section 2)

Around the current trajectory the dynamics factor is replaced by its
linearization, a linear-Gaussian factor. The rewards are already quadratic.
The backward pass is then the elimination of Chapter 6: average over the
next state, maximize over the action. The maximum is put back as a
deterministic policy factor, as in Chapter 4.

In [4]:
I = np.eye(1)
zero = np.zeros(1)


def noise_model(variance):
    """Gaussian noise of the given variance; a hard constraint if it is 0."""
    if variance == 0:
        return noiseModel.Constrained.All(1)
    return noiseModel.Isotropic.Variance(1, variance)


def gaussian(*args):
    """Lift a Gaussian factor to (p, 0)."""
    return SemiringGaussianFactor(JacobianFactor(*args))


def penalty(key):
    """Lift the penalty z^2 on one variable to the reward (1, -z^2)."""
    return SemiringGaussianFactor.Cost(HessianFactor(key, 2 * I, zero, 0.0))


def ordering(key):
    """An ordering holding one key."""
    result = Ordering()
    result.push_back(key)
    return result


def stage1(states, controls, noise):
    """One backward pass on the graph linearized around a trajectory.

    Returns the gains K_t and offsets o_t of the best local policy
    u = -K_t x + o_t, and the value factor (1, V_0) left on the first state.
    """
    moves = len(controls)
    gains, offsets = np.zeros(moves), np.zeros(moves)
    value = penalty(X(moves))  # (1, V_T)
    for t in reversed(range(moves)):
        # Linearized dynamics: x' = x + B u + constant, with B = df/du.
        B = f_u(controls[t])
        constant = states[t + 1] - states[t] - B * controls[t]
        dynamics = gaussian(X(t + 1), I, X(t), -I, U(t), -B * I,
                            np.array([constant]), noise_model(noise))
        # Eliminate the next state by average.
        phi = dynamics.multiply(value).sum(ordering(X(t + 1)))
        bucket = penalty(X(t)).multiply(penalty(U(t))).multiply(phi)
        # Eliminate the action by max: the vertex of the quadratic Q_t in u.
        Q = bucket.value()  # value = 1/2 z'Gz - g'z + f/2, z = (keys)
        keys, G, g = list(Q.keys()), Q.information(), np.ravel(Q.linearTerm())
        u, x = keys.index(U(t)), keys.index(X(t))
        gains[t], offsets[t] = G[u, x] / G[u, u], g[u] / G[u, u]
        best = gaussian(U(t), I, X(t), gains[t] * I, np.array([offsets[t]]),
                        noiseModel.Constrained.All(1))
        value = best.multiply(bucket).sum(ordering(U(t)))  # (1, V_t)
    return gains, offsets, value


def value_at(value, x0):
    """Read V_0(x0) from the value factor on the first state."""
    point = VectorValues()
    point.insert(X(0), np.array([x0]))
    return value.value(point)


controls = np.zeros(T)
states = rollout(controls, x_start)
gains, offsets, value = stage1(states, controls, noise=0.0)
print("first backward pass, around the do-nothing trajectory")
print("gains K_t  =", gains)
print("offsets o_t =", offsets)
print("local model's value at the start:", value_at(value, x_start))
assert np.allclose(gains, [0.6176, 0.6154, 0.6, 0.5], atol=1e-4)

first backward pass, around the do-nothing trajectory
gains K_t  = [0.6176 0.6154 0.6    0.5   ]
offsets o_t = [-0. -0. -0. -0.]
local model's value at the start: -14.558823529411766


## Stage 2: roll out, search along the step, relinearize (Section 3)

In [5]:
def forward(states, controls, gains, offsets, step, x0):
    """Roll out the local policy, with the planned change scaled by step."""
    x, new_controls = x0, np.zeros(len(controls))
    for t in range(len(controls)):
        change = offsets[t] - gains[t] * states[t] - controls[t]
        new_controls[t] = (controls[t] + step * change
                           - gains[t] * (x - states[t]))
        x = f(x, new_controls[t])
    return new_controls


def ilqr(x0, moves, backward, verbose=False, tolerance=1e-8):
    """Relinearize and repeat until the return stops improving.

    Returns the controls and the return before each backward pass.
    """
    controls, history = np.zeros(moves), []
    for iteration in range(100):
        states = rollout(controls, x0)
        current = total_reward(controls, x0)
        history.append(current)
        gains, offsets = backward(states, controls)[:2]
        step = 1.0
        while True:  # halve the step until the return improves
            candidate = forward(states, controls, gains, offsets, step, x0)
            improvement = total_reward(candidate, x0) - current
            if improvement > 0 or step < 1e-8:
                break
            step /= 2
        if verbose:
            print(f"pass {iteration + 1:2d}: return before {current:11.6f}, "
                  f"step {step:g}, improvement {max(improvement, 0):.2e}")
        if improvement > 0:
            controls = candidate
        if improvement < tolerance:
            break
    return controls, history


controls, history = ilqr(
    x_start, T, lambda s, c: stage1(s, c, noise=0.0), verbose=True)
states = rollout(controls, x_start)
print("controls:", controls)
print("states:  ", states)
print("return:  ", total_reward(controls, x_start))

# An independent check with a general-purpose optimizer.
check = minimize(lambda c: -total_reward(c, x_start), np.zeros(T), tol=1e-12)
print("scipy:   ", check.x, -check.fun)
assert np.isclose(total_reward(controls, x_start), -19.549965, atol=1e-5)
assert np.allclose(controls, check.x, atol=1e-3)
assert np.allclose(controls, [-1.2701, -1.0257, -0.7213, -0.3610], atol=1e-3)
print("backward passes:", len(history))
assert len(history) == 13
ilqr_controls, ilqr_states, ilqr_history = controls, states, history

pass  1: return before  -45.000000, step 1, improvement 2.43e+01
pass  2: return before  -20.675198, step 0.5, improvement 1.07e+00
pass  3: return before  -19.604586, step 0.5, improvement 4.54e-02
pass  4: return before  -19.559191, step 0.5, improvement 7.27e-03
pass  5: return before  -19.551921, step 0.5, improvement 1.54e-03
pass  6: return before  -19.550386, step 0.5, improvement 3.30e-04
pass  7: return before  -19.550056, step 0.5, improvement 7.15e-05
pass  8: return before  -19.549985, step 0.5, improvement 1.55e-05
pass  9: return before  -19.549969, step 0.5, improvement 3.35e-06
pass 10: return before  -19.549966, step 0.5, improvement 7.27e-07
pass 11: return before  -19.549965, step 0.5, improvement 1.57e-07
pass 12: return before  -19.549965, step 0.5, improvement 3.41e-08
pass 13: return before  -19.549965, step 0.5, improvement 7.40e-09
controls: [-1.2701 -1.0257 -0.7213 -0.361 ]
states:   [3.     2.1462 1.374  0.7563 0.4102]
return:   -19.549965085370886
scipy:    

## The same backward pass in numpy, and DDP (Sections 2 and 4)

In deviations from the current trajectory, the pass carries the gradient
and the curvature $P_t$ of the value. DDP adds the curvature of the
dynamics to $H_{uu}$.

In [6]:
def backward_numpy(states, controls, ddp=False):
    """Gains and offsets of the local policy; also the curvatures P_t."""
    moves = len(controls)
    gains, offsets = np.zeros(moves), np.zeros(moves)
    P = np.ones(moves + 1)  # V_T = -x^2
    value_gradient = -2 * states[-1]
    for t in reversed(range(moves)):
        B = f_u(controls[t])
        Q_x = -2 * states[t] + value_gradient  # gradient of Q_t in x
        Q_u = -2 * controls[t] + B * value_gradient  # gradient of Q_t in u
        H_uu, H_ux, H_xx = 1 + B * P[t + 1] * B, B * P[t + 1], 1 + P[t + 1]
        if ddp:  # the second derivative of tanh(u) is -2 tanh(u) (1 - tanh^2)
            H_uu -= 0.5 * value_gradient * (-2 * np.tanh(controls[t]) * B)
        gains[t] = H_ux / H_uu
        change = 0.5 * Q_u / H_uu  # the planned change of u_t
        offsets[t] = controls[t] + change + gains[t] * states[t]
        value_gradient = Q_x - gains[t] * Q_u
        P[t] = H_xx - H_ux ** 2 / H_uu
    return gains, offsets, P


# The first pass, around the do-nothing trajectory, in deviation form.
start_states = rollout(np.zeros(T), x_start)
first = backward_numpy(start_states, np.zeros(T))
print("first pass: slopes B_t        =", f_u(np.zeros(T)))
print("            gains K_t         =", first[0])
print("            planned changes   =",
      first[1] - first[0] * start_states[:-1])
assert np.allclose(first[1] - first[0] * start_states[:-1],
                   [-1.8529, -1.8462, -1.8, -1.5], atol=1e-4)

# The module and numpy agree on the local policy.
module = stage1(ilqr_states, ilqr_controls, noise=0.0)
by_hand = backward_numpy(ilqr_states, ilqr_controls)
assert np.allclose(module[0], by_hand[0]) and np.allclose(module[1], by_hand[1])
print("gains at the solution:     ", by_hand[0])
print("curvatures P_t of the value:", by_hand[2])

print("\nDDP:")
ddp_controls, ddp_history = ilqr(
    x_start, T, lambda s, c: backward_numpy(s, c, ddp=True), verbose=True)
assert np.allclose(ddp_controls, ilqr_controls, atol=1e-4)
print("backward passes:", len(ddp_history))
assert len(ddp_history) == 6

first pass: slopes B_t        = [1. 1. 1. 1.]
            gains K_t         = [0.6176 0.6154 0.6    0.5   ]
            planned changes   = [-1.8529 -1.8462 -1.8    -1.5   ]
gains at the solution:      [0.5716 0.604  0.6051 0.496 ]
curvatures P_t of the value: [3.1093 2.496  1.9784 1.5635 1.    ]

DDP:
pass  1: return before  -45.000000, step 1, improvement 2.43e+01
pass  2: return before  -20.675198, step 1, improvement 9.48e-01
pass  3: return before  -19.727300, step 1, improvement 1.75e-01
pass  4: return before  -19.552244, step 1, improvement 2.28e-03
pass  5: return before  -19.549966, step 1, improvement 5.48e-07
pass  6: return before  -19.549965, step 1, improvement 3.55e-14
backward passes: 6


## An exact special case: the line (Section 7)

With linear dynamics the linearization is exact, so one backward pass is the
Riccati recursion of Chapter 6: gains 0.6 and 0.5, and $J^* = -9.25$ for
$x_0 \sim N(2, 1)$.

In [7]:
def stage1_line(noise):
    """The backward pass of stage1 for the line: x' = x + u + noise."""
    gains = np.zeros(2)
    value = penalty(X(2))
    for t in reversed(range(2)):
        dynamics = gaussian(X(t + 1), I, X(t), -I, U(t), -I, zero,
                            noise_model(noise))
        phi = dynamics.multiply(value).sum(ordering(X(t + 1)))
        bucket = penalty(X(t)).multiply(penalty(U(t))).multiply(phi)
        Q = bucket.value()
        keys, G = list(Q.keys()), Q.information()
        gains[t] = G[keys.index(U(t)), keys.index(X(t))] / G[
            keys.index(U(t)), keys.index(U(t))]
        best = gaussian(U(t), I, X(t), gains[t] * I, zero,
                        noiseModel.Constrained.All(1))
        value = best.multiply(bucket).sum(ordering(U(t)))
    return gains, value


line_gains, line_value = stage1_line(noise=0.5)
prior = gaussian(X(0), I, np.array([2.0]), noise_model(1.0))
J_star = prior.multiply(line_value).expectation()
print("gains on the line:", line_gains, " J* =", J_star)
assert np.allclose(line_gains, [0.6, 0.5]) and np.isclose(J_star, -9.25)

gains on the line: [0.6 0.5]  J* = -9.25


## Under noise: three ways to use the solution (Section 6)

Now the moves slip: $x' = x + \tanh(u) + w$ with $w \sim N(0, 0.2)$. The
plan is executed open loop, with the local feedback of the last backward
pass, and by replanning at every step (MPC). The returns are averages over
100000 simulated runs with the same noise.

In [8]:
noise = 0.2
samples = 100000
rng = np.random.default_rng(0)
slips = rng.normal(0.0, np.sqrt(noise), (samples, T))


def simulate(policy):
    """Average return of a policy u = policy(t, x) over the simulated runs."""
    x, total = np.full(samples, x_start), np.zeros(samples)
    for t in range(T):
        u = policy(t, x)
        total -= x ** 2 + u ** 2
        x = f(x, u) + slips[:, t]
    total -= x ** 2
    return total.mean(), total.std() / np.sqrt(samples)


# The local model's own prediction: the noise adds a trace term per move.
local_gains, local_offsets, local_value = stage1(
    ilqr_states, ilqr_controls, noise)
predicted = value_at(local_value, x_start)
print("return predicted by the local model:", predicted)

# MPC: the first control of a fresh plan, tabulated over the state.
grid = np.linspace(-3.0, 6.0, 181)
mpc_table = np.array([
    [ilqr(x, T - t, backward_numpy)[0][0] for x in grid] for t in range(T)])

results = {
    "open loop": simulate(lambda t, x: np.full(samples, ilqr_controls[t])),
    "local feedback": simulate(
        lambda t, x: local_offsets[t] - local_gains[t] * x),
    "MPC": simulate(lambda t, x: np.interp(x, grid, mpc_table[t])),
}
for name, (mean, error) in results.items():
    print(f"{name:15s} return {mean:8.3f} +- {error:.3f}")
assert np.isclose(predicted, -20.958, atol=1e-3)
assert np.isclose(results["open loop"][0], -21.57, atol=0.05)
assert np.isclose(results["local feedback"][0], -21.22, atol=0.05)
assert np.isclose(results["MPC"][0], -21.12, atol=0.05)

return predicted by the local model: -20.957537634885085


open loop       return  -21.568 +- 0.017
local feedback  return  -21.223 +- 0.017
MPC             return  -21.120 +- 0.017


## MAP trajectory optimization with GTSAM (Section 5)

The way a SLAM practitioner would pose the problem: all states and controls
are unknowns of one nonlinear least-squares problem. Each reward is a prior
factor pulling a variable to zero, and each dynamics factor is a soft
constraint with the covariance of the noise.

In [9]:
def dynamics_error(this, values, jacobians):
    """The error x' - f(x, u) of a dynamics factor on (x, u, x')."""
    key_x, key_u, key_next = this.keys()
    x = values.atVector(key_x)[0]
    u = values.atVector(key_u)[0]
    x_next = values.atVector(key_next)[0]
    if jacobians is not None:
        jacobians[0] = np.array([[-1.0]])
        jacobians[1] = np.array([[-f_u(u)]])
        jacobians[2] = np.array([[1.0]])
    return np.array([x_next - f(x, u)])


def map_plan(x0, moves, variance):
    """Maximize the product of all factors over all states and controls."""
    graph = gtsam.NonlinearFactorGraph()
    cost = noiseModel.Isotropic.Variance(1, 0.5)  # error z^2
    graph.add(gtsam.PriorFactorVector(
        X(0), np.array([x0]), noiseModel.Constrained.All(1)))
    for t in range(moves):
        graph.add(gtsam.PriorFactorVector(X(t), zero, cost))
        graph.add(gtsam.PriorFactorVector(U(t), zero, cost))
        graph.add(CustomFactor(noiseModel.Isotropic.Variance(1, variance),
                               [X(t), U(t), X(t + 1)], dynamics_error))
    graph.add(gtsam.PriorFactorVector(X(moves), zero, cost))
    initial = gtsam.Values()
    for t in range(moves + 1):
        initial.insert(X(t), np.array([x0]))
    for t in range(moves):
        initial.insert(U(t), zero)
    parameters = gtsam.LevenbergMarquardtParams()
    parameters.setMaxIterations(500)
    parameters.setRelativeErrorTol(1e-14)
    parameters.setAbsoluteErrorTol(1e-14)
    result = gtsam.LevenbergMarquardtOptimizer(
        graph, initial, parameters).optimize()
    plan_u = np.array([result.atVector(U(t))[0] for t in range(moves)])
    plan_x = np.array([result.atVector(X(t))[0] for t in range(moves + 1)])
    return plan_u, plan_x


def planned_reward(plan_u, plan_x):
    """The return along a planned trajectory, slips included."""
    return -(np.sum(plan_x[:-1] ** 2) + np.sum(plan_u ** 2) + plan_x[-1] ** 2)


# Almost deterministic dynamics: the same trajectory as iLQR.
map_u, map_x = map_plan(x_start, T, 1e-6)
print("MAP, dynamics noise 1e-6: controls", map_u)
print("                          return  ", planned_reward(map_u, map_x))
assert np.allclose(map_u, ilqr_controls, atol=1e-3)

# With the real noise: the plan counts on slips in its favor.
map_u, map_x = map_plan(x_start, T, noise)
assumed_slips = map_x[1:] - f(map_x[:-1], map_u)
print("MAP, dynamics noise 0.2:  controls", map_u)
print("                          states  ", map_x)
print("                          slips it assumes", assumed_slips)
print("                          return it plans ",
      planned_reward(map_u, map_x))
assert np.allclose(map_u, [-0.9654, -0.5814, -0.2489, -0.0776], atol=1e-3)
assert np.isclose(planned_reward(map_u, map_x), -12.573, atol=0.01)

MAP, dynamics noise 1e-6: controls [-1.2701 -1.0257 -0.7213 -0.361 ]
                          return   -19.549845296391045
MAP, dynamics noise 0.2:  controls [-0.9654 -0.5814 -0.2489 -0.0776]
                          states   [3.     1.3806 0.5365 0.1867 0.078 ]
                          slips it assumes [-0.8727 -0.3205 -0.1059 -0.0312]
                          return it plans  -12.572854147068034


MAP inside MPC: replan at every step and apply the first control. The table
of first controls is computed with scipy on the same least-squares problem,
to keep the notebook fast.

In [10]:
def map_first_control(x0, moves, variance):
    def residuals(z):
        u, x = z[:moves], np.concatenate([[x0], z[moves:]])
        return np.concatenate(
            [x, u, (x[1:] - f(x[:-1], u)) / np.sqrt(2 * variance)])
    start = np.concatenate([np.zeros(moves), np.full(moves, x0)])
    return least_squares(residuals, start, xtol=1e-12, ftol=1e-12).x[0]


assert np.isclose(map_first_control(x_start, T, noise), map_u[0], atol=1e-4)
map_table = np.array([
    [map_first_control(x, T - t, noise) for x in grid] for t in range(T)])
results["MAP inside MPC"] = simulate(
    lambda t, x: np.interp(x, grid, map_table[t]))
mean, error = results["MAP inside MPC"]
print(f"MAP inside MPC  return {mean:8.3f} +- {error:.3f}")
print("first control at x = 3:  iLQR", np.interp(3.0, grid, mpc_table[0]),
      " MAP", np.interp(3.0, grid, map_table[0]))
assert np.isclose(mean, -21.80, atol=0.05)

MAP inside MPC  return  -21.799 +- 0.019
first control at x = 3:  iLQR -1.2700913152150128  MAP -0.9653908223214587


## The optimism of MAP, exactly, on the line (Section 5)

On the line everything is in closed form. MAP treats the slip $w$ as a
second control with cost $w^2 / (2 \Sigma_w)$, which here equals $w^2$. The
best split of a total move $m = u + w$ is half each, so the robot commands
only half of the move it plans.

In [11]:
slip_variance = 0.5
P = 1.0  # curvature of MAP's value at the last state
map_gains = np.zeros(2)
for t in reversed(range(2)):
    # Minimize u^2 + w^2/(2 Sigma_w) + P (x + u + w)^2 over u and w.
    H = np.array([[1 + P, P], [P, 1 / (2 * slip_variance) + P]])
    solution = np.linalg.solve(H, [P, P])  # (u, w) = -solution * x
    map_gains[t] = solution[0]
    P = 1 + P - P * (solution[0] + solution[1])
map_belief = -P * 2.0 ** 2  # what MAP expects from x0 = 2
print("MAP gains on the line:", map_gains, " Riccati gains:", line_gains)
print("MAP's optimum from x0 = 2:", map_belief)
# For comparison, with P_0 = 1.6 and beta_0 = 1.25 from the Riccati recursion:
print("best return from x0 = 2 without noise:", -1.6 * 4)
print("best expected return from x0 = 2 with noise:", -(1.6 * 4 + 1.25))
assert map_belief > -1.6 * 4 > -(1.6 * 4 + 1.25)


def evaluate_line(gains):
    """Exact expected return of u = -K_t x on the line, with the module."""
    graph = gtsam.SemiringFactorGraph()
    graph.push_back(gaussian(X(0), I, np.array([2.0]), noise_model(1.0)))
    for t in range(2):
        graph.push_back(gaussian(U(t), I, X(t), gains[t] * I, zero,
                                 noiseModel.Constrained.All(1)))
        graph.push_back(gaussian(X(t + 1), I, X(t), -I, U(t), -I, zero,
                                 noise_model(slip_variance)))
        graph.push_back(penalty(X(t)))
        graph.push_back(penalty(U(t)))
    graph.push_back(penalty(X(2)))
    return graph.expectation()


print("expected return with the Riccati gains:", evaluate_line(line_gains))
print("expected return with the MAP gains:    ", evaluate_line(map_gains))
assert np.allclose(map_gains, [4 / 11, 1 / 3])
assert np.isclose(map_belief, -60 / 11)
assert np.isclose(evaluate_line(line_gains), -9.25)
assert np.isclose(evaluate_line(map_gains), -10.0888, atol=1e-3)

MAP gains on the line: [0.3636 0.3333]  Riccati gains: [0.6 0.5]
MAP's optimum from x0 = 2: -5.454545454545453
best return from x0 = 2 without noise: -6.4
best expected return from x0 = 2 with noise: -7.65
expected return with the Riccati gains: -9.250000000000002
expected return with the MAP gains:     -10.088613406795226


## Numbers for the figures

The trajectories of the first iterations, and the two MPC policies.

In [12]:
controls = np.zeros(T)
for iteration in range(3):
    states = rollout(controls, x_start)
    print(f"after {iteration} passes: controls {controls}")
    print(f"                states   {states}, "
          f"return {total_reward(controls, x_start):.4f}")
    gains, offsets, _ = backward_numpy(states, controls)
    step = 1.0 if iteration == 0 else 0.5
    controls = forward(states, controls, gains, offsets, step, x_start)
print("final:       states", ilqr_states)
for x in [0.0, 1.0, 2.0, 3.0, 4.0]:
    print(f"x = {x}: first control iLQR {np.interp(x, grid, mpc_table[0]):.3f}"
          f"  MAP {np.interp(x, grid, map_table[0]):.3f}")

after 0 passes: controls [0. 0. 0. 0.]
                states   [3. 3. 3. 3. 3.], return -45.0000
after 1 passes: controls [-1.8529 -1.2603 -0.7181 -0.2906]
                states   [3.     2.048  1.1968 0.5811 0.2985], return -20.6752
after 2 passes: controls [-1.1558 -1.0363 -0.79   -0.3681]
                states   [3.     2.1803 1.4039 0.7455 0.3931], return -19.6046
final:       states [3.     2.1462 1.374  0.7563 0.4102]
x = 0.0: first control iLQR 0.000  MAP 0.000
x = 1.0: first control iLQR -0.578  MAP -0.447
x = 2.0: first control iLQR -0.989  MAP -0.759
x = 3.0: first control iLQR -1.270  MAP -0.965
x = 4.0: first control iLQR -1.473  MAP -1.112
